## Limpieza de 'DF_CURSESCAT_SUCIO.xlsx'

Cargamos la tabla de carreras/distancias (una fila por cada distancia ofrecida en cada evento) y le hacemos una primera inspección antes de limpiarla, siguiendo el mismo proceso que en buscametas/championsxip/carreirasgalegas/ccnorte/cronofinisher/mychip/sportmaniacs.

A diferencia de las demás fuentes, esta viene en Excel (no CSV) y esta tabla ya no trae ningún desglose por edad/categoría (solo por distancia y género) — el scraper ya descarta esa información al agregar, así que `publico` aquí prácticamente siempre saldrá "Absoluta/General" (no es un fallo de clasificación, es que la fuente no la ofrece).

In [1]:
from pathlib import Path
import pandas as pd

XLSX_PATH = Path("../../data/raw/cursescat/DF_CURSESCAT_SUCIO.xlsx")
curses = pd.read_excel(XLSX_PATH, sheet_name="Sheet1")

print("Filas x columnas:", curses.shape)
print()
print(curses.dtypes)
curses.head()

Filas x columnas: (500, 13)

nombre_carrera     object
poblacion          object
fecha              object
tipo_carrera       object
Distàncies         object
distancia         float64
URL                object
homes             float64
dones             float64
total             float64
pct_dones         float64
temps_1r           object
estat              object
dtype: object


,nombre_carrera,poblacion,fecha,tipo_carrera,Distàncies,distancia,URL,homes,dones,total,pct_dones,temps_1r,estat
0,Olla de Tapis 2026,Tapis - Maçanet de Cabrenys,2026-06-21,Trail,9km i 21km,9.0,https://www.curses.cat/classificacions/classif...,82.0,56.0,138.0,40.6,00:42:40,OK
1,Olla de Tapis 2026,Tapis - Maçanet de Cabrenys,2026-06-21,Trail,9km i 21km,21.0,https://www.curses.cat/classificacions/classif...,55.0,15.0,70.0,21.4,02:38:29,OK
2,La Cirera 2026,Terrades,2026-06-07,Marxa Cursa Trail,7km i 13km,7.0,https://www.curses.cat/classificacions/classif...,36.0,58.0,94.0,61.7,00:30:36,OK
3,La Cirera 2026,Terrades,2026-06-07,Marxa Cursa Trail,7km i 13km,13.0,https://www.curses.cat/classificacions/classif...,160.0,83.0,243.0,34.2,01:00:14,OK
4,Espardenyada 2026,Sant Jaume de Llierca,2026-06-07,Marxa trail,9km i 16km,9.0,https://www.curses.cat/classificacions/classif...,68.0,70.0,138.0,50.7,00:57:36,OK


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados, estado del scraping y
# rango de fechas. Igual que "estat_esdeveniment" en cronofinisher, aquí
# "estat" marca si el scraper pudo sacar resultados ("OK", a veces con
# nota de categorías descartadas) o no ("URL no processable"/"Sense dades
# web") — se comprueba más abajo.
print("Valores nulos por columna:")
print(curses.isna().sum())
print()

print("Filas completamente duplicadas:", curses.duplicated().sum())
print()

print("Rango de fechas (texto):", curses["fecha"].min(), "->", curses["fecha"].max())
print()

print("estat (top 10):")
print(curses["estat"].value_counts().head(10))
print()

print("Filas con estat que empieza por 'OK':", curses["estat"].str.startswith("OK").sum(), "de", len(curses))

Valores nulos por columna:
nombre_carrera     0
poblacion          2
fecha              0
tipo_carrera       0
Distàncies         0
distancia         31
URL                0
homes             31
dones             31
total             31
pct_dones         31
temps_1r          31
estat              0
dtype: int64

Filas completamente duplicadas: 1

Rango de fechas (texto): 2015-11-11 -> 2026-06-21

estat (top 10):
estat
OK                                                                                         392
URL no processable                                                                          20
Sense dades web                                                                             11
OK (categoria(es) descartada(es): idx=2 (0:38:01), idx=3 (1:10:34))                          4
OK (categoria(es) descartada(es): idx=1 (00:44:25))                                          3
OK (categoria(es) descartada(es): idx=3 (0:52:23), idx=0 (1:05:21))                          3
OK (cate

In [3]:
# Nos quedamos solo con las filas con resultados reales ("estat" empieza
# por "OK" — incluye las que tienen categorías descartadas, que sí traen
# recuentos válidos). Después, quitamos duplicados exactos ANTES de
# seleccionar columnas. Aquí no hay ningún id de fila único (ni race_id
# como en carreirasgalegas, ni event_id como en sportmaniacs) — pero eso
# no significa que valga usar (URL, distancia) como clave: ya hay eventos
# con dos filas de la misma URL+distancia que son categorías legítimas y
# distintas (p.ej. "Crematorrons 2023" 28K tiene una fila 76H/6D y otra
# 21H/0D — dos pruebas distintas a la misma distancia), así que solo
# comparamos filas completas (todas las columnas iguales) para no repetir
# el error de carreirasgalegas.
antes = len(curses)
curses = curses[curses["estat"].str.startswith("OK")].reset_index(drop=True)
print(f"Filas sin datos de resultados descartadas: {antes - len(curses)} ({antes} -> {len(curses)})")

antes = len(curses)
curses = curses.drop_duplicates().reset_index(drop=True)
print(f"{antes - len(curses)} filas duplicadas eliminadas ({antes} -> {len(curses)})")

Filas sin datos de resultados descartadas: 31 (500 -> 469)
0 filas duplicadas eliminadas (469 -> 469)


In [4]:
# Limpieza: nos quedamos con las columnas que interesan, renombradas.
# "poblacion" ya es un municipio limpio (como "lloc" en carreirasgalegas/
# cronofinisher), así que se recupera directamente como "municipio". La
# "distancia" ya viene como número limpio en km (a diferencia de las demás
# fuentes, no hace falta parsear texto). El resto de columnas ("Distàncies"
# —texto redundante con "distancia"—, "URL" —artefacto técnico—, "total"
# —redundante con homes+dones—, "pct_dones", "temps_1r" y "estat") no se
# incluyen.
curses_limpio = curses[
    ["nombre_carrera", "fecha", "poblacion", "tipo_carrera", "distancia", "dones", "homes"]
].rename(columns={
    "poblacion": "municipio",
    "tipo_carrera": "modalidad",
    "dones": "finisher_d",
    "homes": "finisher_h",
})

curses_limpio["fecha"] = pd.to_datetime(curses_limpio["fecha"])
curses_limpio[["finisher_d", "finisher_h"]] = curses_limpio[["finisher_d", "finisher_h"]].fillna(0).astype(int)

print(curses_limpio.shape)
curses_limpio.head()

(469, 7)


,nombre_carrera,fecha,municipio,modalidad,distancia,finisher_d,finisher_h
0,Olla de Tapis 2026,2026-06-21,Tapis - Maçanet de Cabrenys,Trail,9.0,56,82
1,Olla de Tapis 2026,2026-06-21,Tapis - Maçanet de Cabrenys,Trail,21.0,15,55
2,La Cirera 2026,2026-06-07,Terrades,Marxa Cursa Trail,7.0,58,36
3,La Cirera 2026,2026-06-07,Terrades,Marxa Cursa Trail,13.0,83,160
4,Espardenyada 2026,2026-06-07,Sant Jaume de Llierca,Marxa trail,9.0,70,68


In [5]:
# Clasificamos la disciplina por palabras clave en "modalidad" (antes
# "tipo_carrera" — texto en catalán, relativamente limpio: 98 valores
# únicos pero la mayoría son combinaciones de las mismas pocas palabras).
# Natación/travesías a nado, obstáculos y canicross van a "Otros" (deporte
# de una sola disciplina ajena a nuestras categorías), igual que en el
# resto de fuentes. Si "modalidad" tiene texto pero no encaja en ningún
# patrón (p.ej. erratas del scraping como "Croonoescalada"/"Pupolar", o
# "Solidària" sin ninguna disciplina asociada), también va a "Otros" — no
# hay ningún caso aquí en el que "modalidad" esté vacía, así que no hace
# falta un valor por defecto distinto.
import re

_CATEGORIAS = {
    "Otros": r"natac|travess|obstacle|canicross",
    "trail running": r"trail|tral|muntanya|backyard|crono.?escalada",
    "Ciclismo y btt": r"\bbtt\b|\bmtb\b|ciclis|\bbici\b|ciclotur|gravel|gran ?fondo|\bbike\b|e-?bike",
    "marcha": r"marxa|\bruta\b|caminada",
    "road running": r"cursa|popular|pupolar|urbana|corre|running",
}

def _clasificar_texto(texto):
    for categoria, patron in _CATEGORIAS.items():
        if re.search(patron, texto, flags=re.IGNORECASE):
            return categoria
    return None

def _clasificar(row):
    texto_modalidad = row["modalidad"] if isinstance(row["modalidad"], str) else ""
    categoria = _clasificar_texto(texto_modalidad) if texto_modalidad else None
    return categoria if categoria else "Otros"

curses_limpio["tipo_modalidad"] = curses_limpio.apply(_clasificar, axis=1)

print(curses_limpio["tipo_modalidad"].value_counts())
print()
print("Texto de modalidad que ha caído en 'Otros' sin ser natación/obstáculos/canicross (revisar):")
_otros_conocidos = curses_limpio["modalidad"].str.contains(
    r"natac|travess|obstacle|canicross", case=False, regex=True, na=False
)
print(curses_limpio.loc[(curses_limpio["tipo_modalidad"] == "Otros") & ~_otros_conocidos, "modalidad"].value_counts())

tipo_modalidad
trail running     268
marcha             72
road running       59
Ciclismo y btt     37
Otros              33
Name: count, dtype: int64

Texto de modalidad que ha caído en 'Otros' sin ser natación/obstáculos/canicross (revisar):
modalidad
Solidària         7
Trial             2
Croonoescalada    1
Solidari          1
Name: count, dtype: int64


In [ ]:
# Clasificamos el público (edad) por palabras clave, igual que en el resto
# de fuentes — pero esta tabla ya no trae ningún desglose por categoría de
# edad (SUB/infantil/veterano...), solo por distancia y género: el scraper
# los agrega/descarta en el proceso (ver "estat" == "OK (categoria(es)
# descartada(es)...)"). Así que, salvo alguna coincidencia puntual en el
# nombre de la carrera, esto va a salir prácticamente 100% "Absoluta/
# General" — no es un fallo del código, es que la información no existe
# en esta fuente. Cadete/juvenil se funde en "Infantil" (igual que en
# carreirasgalegas/ccnorte) para que 'Infantil' signifique lo mismo en
# las 11 fuentes al unirlas en Limpieza_union.ipynb.
_OTROS_PATRON = r"discapac|invident|handbike|silla de ruedas|adaptad"
_EQUIPOS_PATRON = r"equipos?\b|equips?\b"

_PUBLICOS_TEXTO = {
    "Elite": r"\belit|professional|profesional",
    "Mayores/Veteranos": r"veter|master|m[aà]ster",
}
_INFANTIL_PATRON = r"infantil|alev[ií]n|benjam|prebenjam|escolar|ni[nñ]os|a[nñ]os|cadet|juvenil|junior"

def _clasificar_publico_texto(texto):
    if re.search(_EQUIPOS_PATRON, texto, flags=re.IGNORECASE):
        return "Equipos"
    if re.search(_OTROS_PATRON, texto, flags=re.IGNORECASE):
        return "Otros"
    for publico, patron in _PUBLICOS_TEXTO.items():
        if re.search(patron, texto, flags=re.IGNORECASE):
            return publico
    if re.search(_INFANTIL_PATRON, texto, flags=re.IGNORECASE):
        return "Infantil"
    return None

def _clasificar_publico(row):
    texto_modalidad = row["modalidad"] if isinstance(row["modalidad"], str) else ""
    publico = _clasificar_publico_texto(texto_modalidad) if texto_modalidad else None
    if publico:
        return publico

    texto_nombre = row["nombre_carrera"] if isinstance(row["nombre_carrera"], str) else ""
    publico = _clasificar_publico_texto(texto_nombre)
    if publico:
        return publico

    return "Absoluta/General"

curses_limpio["publico"] = curses_limpio.apply(_clasificar_publico, axis=1)

print(curses_limpio["publico"].value_counts())

In [7]:
# "distancia" ya venía numérica y limpia (km), pero con algunos huecos en
# las filas de categorías descartadas — los rellenamos a 0, igual que en
# el resto de fuentes. "modalidad" ya ha cumplido su función (tipo_modalidad
# y publico), así que la quitamos, igual que en buscametas/championsxip.
curses_limpio["distancia"] = curses_limpio["distancia"].fillna(0)
curses_limpio = curses_limpio.drop(columns=["modalidad"])

print("Filas con distancia detectada:", (curses_limpio["distancia"] != 0).sum(),
      "de", len(curses_limpio))
curses_limpio.columns.tolist()

Filas con distancia detectada: 448 de 469


['nombre_carrera',
 'fecha',
 'municipio',
 'distancia',
 'finisher_d',
 'finisher_h',
 'tipo_modalidad',
 'publico']

### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, championsxip, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia`. `fuente` es una constante ("cursescat") para identificar el origen al concatenar las tablas. `municipio` se recupera de `poblacion` (antes se descartaba); `comarca`/`provincia` no vienen en la fuente, así que las geocodificamos a partir de `municipio` (igual que en carreirasgalegas/iter5/cruzandolameta), asumiendo Catalunya para desambiguar (todos los eventos de curses.cat están en esa comunidad). Esta fuente no tiene ninguna columna propia que valga la pena conservar (ni id, ni desglose de categoría), así que el esquema final queda con exactamente estas 12 columnas, igual que en championsxip.

In [8]:
# Geocodificamos "comarca"/"provincia" a partir de "municipio" (igual que
# en carreirasgalegas): aquí ya es un nombre de municipio limpio (aunque
# con variantes de mayúsculas/typos entre eventos distintos del mismo
# lugar — no las normalizamos, cada variante se geocodifica por separado,
# el resultado es el mismo). Solo ~100 municipios únicos, así que es
# rápido. Checkpoint propio en cursescat_ubicaciones.csv. Asumimos
# Catalunya para desambiguar (todos los eventos de curses.cat están en
# esta comunidad, sobre todo comarques gironines).
import csv
import time


def geocodificar_ubicacion_municipios(municipios, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_ubic = out_path / "cursescat_ubicaciones.csv"

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["municipio"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} municipios ya geocodificados")

    geolocator = Nominatim(user_agent="cursescat_ubicaciones_claudia")

    municipios_unicos = list(dict.fromkeys(m for m in municipios if isinstance(m, str)))
    pendientes = [m for m in municipios_unicos if m not in cache]
    print(f"Municipios a geocodificar: {len(pendientes)} (de {len(municipios_unicos)} únicos)")

    campos = ["municipio", "comarca", "provincia"]
    write_header = not csv_ubic.exists()
    with open(csv_ubic, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        if write_header:
            writer.writeheader()

        for i, municipio in enumerate(pendientes, 1):
            fila = {"municipio": municipio, "comarca": None, "provincia": None}
            try:
                loc = geolocator.geocode(
                    f"{municipio}, Catalunya, España", exactly_one=True, country_codes="es",
                    addressdetails=True, timeout=10,
                )
                if loc:
                    addr = loc.raw.get("address", {})
                    fila["comarca"] = addr.get("county")
                    fila["provincia"] = addr.get("province") or addr.get("state")
            except GeopyError as e:
                print(f"  [{municipio}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{municipio}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[municipio] = fila

            if i % 25 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")
            time.sleep(pausa_segundos)

    print(f"CSV de ubicaciones: {csv_ubic.resolve()}")
    return cache


OUT_DIR = Path("../../data/raw/cursescat")
_ubicaciones = geocodificar_ubicacion_municipios(curses_limpio["municipio"], out_dir=OUT_DIR)
curses_limpio["comarca"] = curses_limpio["municipio"].map(lambda m: _ubicaciones.get(m, {}).get("comarca"))
curses_limpio["provincia"] = curses_limpio["municipio"].map(lambda m: _ubicaciones.get(m, {}).get("provincia"))

print("Filas con provincia:", curses_limpio["provincia"].notna().sum(), "de", len(curses_limpio))
print("Filas con comarca:", curses_limpio["comarca"].notna().sum(), "de", len(curses_limpio))
curses_limpio[["municipio", "comarca", "provincia"]].drop_duplicates().sample(15, random_state=0)

Checkpoint: 101 municipios ya geocodificados
Municipios a geocodificar: 0 (de 101 únicos)
CSV de ubicaciones: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\cursescat_data\cursescat_ubicaciones.csv
Filas con provincia: 465 de 469
Filas con comarca: 465 de 469


,municipio,comarca,provincia
51,Llagostera,Gironès,Girona
378,GIRONA,Gironès,Girona
4,Sant Jaume de Llierca,Garrotxa,Girona
165,Cassà de la Selva,Gironès,Girona
376,ORDIS,Alt Empordà,Girona
361,Port de la Selva,la Selva,Girona
32,Figueres,Alt Empordà,Girona
219,SANT LLORENÇ DE LA MUGA,Alt Empordà,Girona
159,Cim de Sant Miquel,Vallès Oriental,Catalunya
226,Palau-Saverdera,Alt Empordà,Girona


In [9]:
# Añadimos "fuente" (constante, para identificar el origen al concatenar
# con las otras 8 tablas) y "dia_semana" (derivado de "fecha"), y
# reordenamos las columnas para que el esquema común (fuente,
# nombre_carrera, fecha, dia_semana, distancia, tipo_modalidad, publico,
# finisher_d, finisher_h, municipio, comarca, provincia) quede igual en
# las 9 fuentes. Aquí no hay ninguna columna propia que añadir al final.
curses_limpio["fuente"] = "cursescat"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia"]
]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia']

In [10]:
# Vista final de la tabla ya limpia y clasificada
print("Columnas:", list(curses_limpio.columns))
print("Filas x columnas:", curses_limpio.shape)
print()
print(curses_limpio.dtypes)
print()
print("Cruce tipo_modalidad x publico:")
print(pd.crosstab(curses_limpio["tipo_modalidad"], curses_limpio["publico"]))
print()
curses_limpio.sample(15)

Columnas: ['fuente', 'nombre_carrera', 'fecha', 'dia_semana', 'distancia', 'tipo_modalidad', 'publico', 'finisher_d', 'finisher_h', 'municipio', 'comarca', 'provincia']
Filas x columnas: (469, 12)

fuente                    object
nombre_carrera            object
fecha             datetime64[ns]
dia_semana                object
distancia                float64
tipo_modalidad            object
publico                   object
finisher_d                 int64
finisher_h                 int64
municipio                 object
comarca                   object
provincia                 object
dtype: object

Cruce tipo_modalidad x publico:
publico         Absoluta/General
tipo_modalidad                  
Ciclismo y btt                37
Otros                         33
marcha                        72
road running                  59
trail running                268



,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia
152,cursescat,Trail La Vall,2024-12-01,Domingo,22.0,trail running,Absoluta/General,18,87,Bellcaire d'Empordà,Baix Empordà,Girona
8,cursescat,11ª Edició Marxa de l'Ordi,2026-05-31,Domingo,10.0,marcha,Absoluta/General,49,89,Ordis,Alt Empordà,Girona
163,cursescat,Marxa Trail LLuerts,2024-11-10,Domingo,8.0,trail running,Absoluta/General,63,50,Viladamat,Alt Empordà,Girona
104,cursescat,Marxa Ordis 2025,2025-05-25,Domingo,13.0,marcha,Absoluta/General,81,106,Ordis,Alt Empordà,Girona
196,cursescat,La Cirera,2024-06-02,Domingo,19.0,trail running,Absoluta/General,17,53,Terrades,Alt Empordà,Girona
408,cursescat,Sant Esteve Llagostera,2021-12-26,Domingo,5.0,marcha,Absoluta/General,45,150,LLagostera,Gironès,Girona
268,cursescat,BesalúCross,2023-10-29,Domingo,20.0,trail running,Absoluta/General,15,65,Besalú,Garrotxa,Girona
397,cursescat,BALCÓ DE L'EMPORDÀ,2022-02-13,Domingo,21.0,trail running,Absoluta/General,18,86,PALAU-SAVERDERA,Alt Empordà,Girona
30,cursescat,La Marreca,2026-04-11,Sábado,5.0,marcha,Absoluta/General,13,13,Salt,Gironès,Girona
328,cursescat,Sant Esteve Llagostera,2022-12-26,Lunes,10.0,marcha,Absoluta/General,58,138,Llagostera,Gironès,Girona


In [11]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/cursescat/DF_CURSESCAT_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\cursescat_data\DF_CURSESCAT_LIMPIO.csv